# Training and Evaluation (Aligned with `src/train.py` and `src/evaluate.py`)

This notebook mirrors the production model pipeline exactly:
- load preprocessed train/validation/test artifacts
- train with class weights and early stopping on `val_auc`
- evaluate with ROC AUC, PR AUC, precision, recall, F1, confusion matrix
- save model and metrics to `artifacts/model/`

In [1]:
import json
import random
from pathlib import Path

import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

I0000 00:00:1778064846.930069   23290 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
SEED = 42
BATCH_SIZE = 128
EPOCHS = 100
PATIENCE = 5
LEARNING_RATE = 1e-3
HIDDEN_SIZE = 64
THRESHOLD = 0.5

DATA_DIR = Path("artifacts/data")
MODEL_DIR = Path("artifacts/model")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [3]:
def load_split(path: Path):
    data = np.load(path)
    return data["inputs"].astype(np.float32), data["targets"].astype(np.int32)


train_x, train_y = load_split(DATA_DIR / "train.npz")
val_x, val_y = load_split(DATA_DIR / "validation.npz")
test_x, test_y = load_split(DATA_DIR / "test.npz")

print(train_x.shape, val_x.shape, test_x.shape)

(11266, 10) (1409, 10) (1409, 10)


In [4]:
pos = int(train_y.sum())
neg = len(train_y) - pos
class_weight = {
    0: len(train_y) / (2.0 * max(neg, 1)),
    1: len(train_y) / (2.0 * max(pos, 1)),
}
class_weight

{0: 0.594386409201224, 1: 3.1486864169927333}

In [5]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(train_x.shape[1],)),
    tf.keras.layers.Dense(HIDDEN_SIZE, activation="relu"),
    tf.keras.layers.Dense(HIDDEN_SIZE, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy"), tf.keras.metrics.AUC(name="auc")],
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_auc",
    patience=PATIENCE,
    mode="max",
    restore_best_weights=True,
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=str(MODEL_DIR / "best_model.keras"),
    monitor="val_auc",
    mode="max",
    save_best_only=True,
)

history = model.fit(
    train_x,
    train_y,
    validation_data=(val_x, val_y),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=[early_stopping, checkpoint],
    verbose=2,
)

Epoch 1/100


E0000 00:00:1778064850.911224   23290 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


89/89 - 1s - 16ms/step - accuracy: 0.7180 - auc: 0.8209 - loss: 0.5325 - val_accuracy: 0.8460 - val_auc: 0.8689 - val_loss: 0.4152
Epoch 2/100
89/89 - 0s - 2ms/step - accuracy: 0.8197 - auc: 0.8755 - loss: 0.4248 - val_accuracy: 0.8573 - val_auc: 0.8893 - val_loss: 0.3777
Epoch 3/100
89/89 - 0s - 2ms/step - accuracy: 0.8301 - auc: 0.8873 - loss: 0.3990 - val_accuracy: 0.8573 - val_auc: 0.8961 - val_loss: 0.3574
Epoch 4/100
89/89 - 0s - 2ms/step - accuracy: 0.8368 - auc: 0.8929 - loss: 0.3866 - val_accuracy: 0.8602 - val_auc: 0.8997 - val_loss: 0.3502
Epoch 5/100
89/89 - 0s - 2ms/step - accuracy: 0.8412 - auc: 0.8967 - loss: 0.3784 - val_accuracy: 0.8602 - val_auc: 0.9021 - val_loss: 0.3475
Epoch 6/100
89/89 - 0s - 2ms/step - accuracy: 0.8445 - auc: 0.8994 - loss: 0.3727 - val_accuracy: 0.8623 - val_auc: 0.9049 - val_loss: 0.3420
Epoch 7/100
89/89 - 0s - 2ms/step - accuracy: 0.8437 - auc: 0.9011 - loss: 0.3685 - val_accuracy: 0.8637 - val_auc: 0.9061 - val_loss: 0.3370
Epoch 8/100
89/89

In [6]:
history_path = MODEL_DIR / "history.json"
config_path = MODEL_DIR / "train_config.json"

with history_path.open("w", encoding="utf-8") as fp:
    json.dump(history.history, fp, indent=2)

config = {
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "learning_rate": LEARNING_RATE,
    "hidden_size": HIDDEN_SIZE,
    "seed": SEED,
    "class_weight": class_weight,
}
with config_path.open("w", encoding="utf-8") as fp:
    json.dump(config, fp, indent=2)

print(history_path)
print(config_path)

artifacts/model/history.json
artifacts/model/train_config.json


In [7]:
best_model = tf.keras.models.load_model(MODEL_DIR / "best_model.keras")
probs = best_model.predict(test_x, verbose=0).reshape(-1)
preds = (probs >= THRESHOLD).astype(np.int32)

metrics = {
    "threshold": THRESHOLD,
    "roc_auc": float(roc_auc_score(test_y, probs)),
    "pr_auc": float(average_precision_score(test_y, probs)),
    "precision": float(precision_score(test_y, preds, zero_division=0)),
    "recall": float(recall_score(test_y, preds, zero_division=0)),
    "f1": float(f1_score(test_y, preds, zero_division=0)),
    "confusion_matrix": confusion_matrix(test_y, preds).tolist(),
    "classification_report": classification_report(test_y, preds, zero_division=0, output_dict=True),
}

metrics

{'threshold': 0.5,
 'roc_auc': 0.8999039330922242,
 'pr_auc': 0.7493754076836128,
 'precision': 0.5604395604395604,
 'recall': 0.6830357142857143,
 'f1': 0.6156941649899397,
 'confusion_matrix': [[1065, 120], [71, 153]],
 'classification_report': {'0': {'precision': 0.9375,
   'recall': 0.8987341772151899,
   'f1-score': 0.9177078845325292,
   'support': 1185.0},
  '1': {'precision': 0.5604395604395604,
   'recall': 0.6830357142857143,
   'f1-score': 0.6156941649899397,
   'support': 224.0},
  'accuracy': 0.8644428672817601,
  'macro avg': {'precision': 0.7489697802197802,
   'recall': 0.7908849457504521,
   'f1-score': 0.7667010247612345,
   'support': 1409.0},
  'weighted avg': {'precision': 0.8775556859747774,
   'recall': 0.8644428672817601,
   'f1-score': 0.8696943478557796,
   'support': 1409.0}}}

In [8]:
eval_path = MODEL_DIR / "evaluation.json"
with eval_path.open("w", encoding="utf-8") as fp:
    json.dump(metrics, fp, indent=2)

print(
    f"ROC AUC={metrics['roc_auc']:.4f} | PR AUC={metrics['pr_auc']:.4f} | "
    f"Precision={metrics['precision']:.4f} | Recall={metrics['recall']:.4f} | F1={metrics['f1']:.4f}"
)
print("Confusion matrix [ [TN, FP], [FN, TP] ]:", metrics["confusion_matrix"])
print(f"Saved evaluation report to: {eval_path.resolve()}")

ROC AUC=0.8999 | PR AUC=0.7494 | Precision=0.5604 | Recall=0.6830 | F1=0.6157
Confusion matrix [ [TN, FP], [FN, TP] ]: [[1065, 120], [71, 153]]
Saved evaluation report to: /teamspace/studios/this_studio/Book_puch_pred/artifacts/model/evaluation.json
